In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d


from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst

import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

# Read in the file

In [ ]:
for d in [-7.5, 10, 100, 1000]:
    for r in [500, 4000]:
        tag = f'd_100_r_{r}_mDM_200-10000_mA_0.22_dm_model_floating'
        #tag = 'd_100_r_4000_mDM_200-10000_mA_0.22_dm_model_floating'
        
        infile = f'generated_data_{tag}_HIT_DETECTOR_COMBINED.parquet'
        df_decays = pd.read_parquet(infile)
        
        for mass in df_decays['M_DM'].unique():
            #mass = 1000
            
            filter = (df_decays['efinal_mu1']>10)
            filter = filter & (df_decays['M_DM']==mass)
            
            plt.figure(figsize=(12,4))
            
            plt.subplot(1,3,1)
            plt.hist(-df_decays[filter]['z0'],bins=100, range=(-4000,0))
            plt.xlabel('depth (m)', fontsize=18)
            
            plt.subplot(1,3,2)
            #df_decays[filter]['efinal_mu1'].hist(bins=50, range=(-100,8000))
            df_decays[filter]['pt1_detector_acceptance_eloss'].hist(bins=50, range=(-100,8000))
            plt.xlabel(r'$\mu  \; p_{T}$ at detector (GeV)', fontsize=18)
            
            plt.subplot(1,3,3)
            #df_decays[filter].plot.scatter(y='efinal_mu1', x='y0', s=0.1, ax=plt.gca())
            df_decays[filter].plot.scatter(y='pt1_detector_acceptance_eloss', x='y0', s=0.1, ax=plt.gca())
            
            DMstr = 'DM'
            lbracket = '{'
            rbracket = '}'
            plt.gcf().suptitle(f'$M_{lbracket}DM{rbracket}$ {int(mass)} GeV/c$^2$')
            
            plt.tight_layout()
            
            outfile = 'depth_and_pt_d_{r}_r_{r}_{tag}.png'
            plt.savefig(outfile)

In [ ]:
for d in [-7.5, 10, 100, 1000]:
    for r in [500, 4000]:
#for d in [1000]:
#    for r in [4000]:

        tag = f'd_{d}_r_{r}_mDM_200-10000_mA_0.22_dm_model_floating'
        #tag = 'd_100_r_4000_mDM_200-10000_mA_0.22_dm_model_floating'
        
        infile = f'generated_data_{tag}_HIT_DETECTOR_COMBINED.parquet'
        df_decays = pd.read_parquet(infile)
            
        plt.figure(figsize=(12,6))
    
        for idx, mass in enumerate(df_decays['M_DM'].unique()):
            #mass = 1000
            
            filter = (df_decays['efinal_mu1']>10)
            filter = filter & (df_decays['M_DM']==mass)
                        
            plt.subplot(2,4,idx+1)

            DMstr = 'DM'
            lbracket = '{'
            rbracket = '}'

            label = f'$M_{lbracket}DM{rbracket}$ = {int(mass)} GeV/c$^2$'

            x = df_decays[filter]['pt1_detector_acceptance_eloss']
            sumx = len(x)
            if sumx == 0:
                sumx = 1
            weights = (1/sumx)*np.ones_like(x)

            plt.hist(x, weights=weights, bins=80, range=(0,8000),label=label)
            plt.xlabel(r'$\mu  \; p_{T}$ at detector (GeV)', fontsize=10)
            #plt.yscale('log')
            plt.legend()

            #if d==-7.5:
            #    d = 0

            plt.gcf().suptitle(f'radius = {r} m   depth = {d} m')
            
            plt.tight_layout()
            
        outfile = f'pt_{tag}_NOLOGSCALE.png'
        plt.savefig(outfile)

In [ ]:

#tag = f'd_{d}_r_{r}_mDM_200-10000_mA_0.22_dm_model_floating'
#tag = 'd_100_r_4000_mDM_200-10000_mA_0.22_dm_model_floating'

#infile = 'Generated data/generated_data_d_-7.5-4000_r_4000_mDM_200-10000_mA_0.22_dm_model_core_HIT_DETECTOR_COMBINED.parquet'
#infile = 'generated_data_d__r_458_mDM_2000-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet'
infile = 'generated_data_d_-1007.5_r_458_mDM_2000-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet'
#infile = 'generated_data_d_-107.5_r_458_mDM_2000-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet'
#infile = 'generated_data_d_-7.6_r_458_mDM_2000-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet'

tag = f'muon_pT_at_detector_{infile.split("_HIT_DETECTOR_COMBINED.parquet")[0]}'

#infile = f'generated_data_{tag}_HIT_DETECTOR_COMBINED.parquet'
df_decays = pd.read_parquet(infile)

print(df_decays['M_DM'].unique())

plt.figure(figsize=(12,6))

#masses = np.array([  200.,  1000.,  1600. , 2000. , 3000. , 6000. , 7000. ,10000.])
#masses = np.array([  200.,  1000.,   3000. , 7000. ,10000.])

masses = df_decays['M_DM'].unique()

for idx, mass in enumerate(masses):
    #mass = 1000
    
    filter = (df_decays['efinal_mu1']>10)
    filter = filter & (df_decays['M_DM']==mass)
                
#    plt.subplot(2,4,idx+1)

    DMstr = 'DM'
    lbracket = '{'
    rbracket = '}'

    label = f'$M_{lbracket}DM{rbracket}$ = {int(mass)} GeV/c$^2$'

    x = df_decays[filter]['pt1_detector_acceptance_eloss']
    sumx = len(x)
    if sumx == 0:
        sumx = 1
    weights = (1/sumx)*np.ones_like(x)

    plt.hist(x, weights=weights, bins=75, range=(0,12000),label=label, density=True, histtype='step', linewidth=3)
    plt.xlabel(r'$\mu  \; p_{T}$ at detector (GeV)', fontsize=18)
    plt.ylabel(r'arb. units', fontsize=18)

    plt.yscale('log')
    plt.legend(fontsize=18)

    #if d==-7.5:
    #    d = 0

    #plt.gcf().suptitle(f'radius = {r} m   depth = {d} m')
    
plt.tight_layout()
    
outfile = f'{tag}_LOGSCALE.png'
plt.savefig(outfile)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

#df = pd.read_parquet("/Users/sanjitmasanam/Documents/CodingProjects/CMS/t0timing_chamber_study/EarthShine_depth_samples/generated_data_d_-1007.5_r_458_mDM_2000-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet")
df = pd.read_parquet(infile)


df_mass = df['M_DM'].to_numpy()
df_pT = df['pt1_detector_acceptance_eloss'].to_numpy()

# Dark matter masses (GeV)
masses = np.arange(2000, 11000, 1000)

# Binning (matches the visible range)
bins = np.linspace(0, 4000, 25)

plt.figure(figsize=(12, 6))

for M in masses:
    # --- Synthetic pT spectrum ---
    # Mean pT increases with DM mass
    scale = 0.25 * M
    pt = df_pT[(np.isclose(df_mass, M))]

    # Physical cutoff (roughly matching the figure)
    pt = pt[pt < 0.4 * M]

    # Histogram (normalized, step-style)
    plt.hist(
        pt,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2.5,
        label=rf"$M_{{DM}}$ = {M} GeV/$c^2$"
    )

# Axes styling
plt.yscale("log")
plt.xlabel(r"$\mu\ p_T$ (GeV) at detector w/ starting depth of -1007.5 m", fontsize=14)
plt.ylabel("arb. units", fontsize=14)

# Limits to match visual appearance
plt.xlim(0, 12500)
plt.ylim(3e-7, 4e-2)

# Legend
plt.legend(
    fontsize=13,
    frameon=True,
    loc="upper right"
)

plt.tight_layout()
plt.show()

In [ ]:
df_mass = df['M_DM'].to_numpy()
df_pT = df['pt1_detector_acceptance'].to_numpy()


M = 10000

pt = df_pT[(np.isclose(df_mass, M))]

# Physical cutoff (roughly matching the figure)
#pt = pt[pt < 0.4 * M]
pt = pt[pt<M]

In [ ]:
#plt.hist(df_pT,bins=100);
plt.hist(pt,bins=100);

In [ ]:
bins = np.linspace(0, 4000, 25)
len(bins)